# Workspace Setup (Admin-only)

This notebook provisions the shared resources for **Deploying and Monitoring Agents on Databricks**. Run it once per workspace, as a workspace admin, before learners begin. It is idempotent: re-running it keeps existing services and recreates any whose model changed.

**What it creates (all in `workspace.agent_deployment`):**
1. Two Unity Gateway model services that learners use as a shared resource. The model each one routes to is set in `config/config-workspace-setup.yaml`:
    - `agent-model`: the model the agents and each learner's app call
    - `judge-model`: the model for the scorers and judges
2. `EXECUTE` on the schema (the model services) for all users

Each learner deploys their own app in the course. The app's bundle grants its service principal `EXECUTE` on `agent-model`.

**Prerequisites:** the `dbacademy` catalog.

Run on **Serverless V6**.

## Install Libraries

In [0]:
import sys, os

sys.path.insert(0, os.path.abspath('.'))
config_path = "./config/config-workspace-setup.yaml"
from _lib.library_installer import install_libraries
install_libraries(config_path)

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys

sys.path.insert(0, os.path.abspath('.'))
config_path = "./config/config-workspace-setup.yaml"

## 1. Define Variables and Create the Shared Schema

In [0]:
SHARED_CATALOG = "workspace"  # this repo: course default "dbacademy" (Academy lab catalog)
SHARED_SCHEMA  = "agent_deployment"
SHARED         = f"{SHARED_CATALOG}.{SHARED_SCHEMA}"

# Unity Gateway model services and the models they route to, from
# config/config-workspace-setup.yaml (update models there, not here)
from _lib.config_loader import load_config

setup_cfg = load_config(config_path)
MODEL_SERVICES = setup_cfg.get("model_services") or {}
RECREATE_ON_DRIFT = bool(setup_cfg.get("recreate_model_services_on_drift", False))

# This repo: only create the catalog if it is missing. CREATE CATALOG IF NOT EXISTS fails on
# metastores without a storage root even when the catalog already exists.
if SHARED_CATALOG not in [r[0] for r in spark.sql("SHOW CATALOGS").collect()]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {SHARED_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED}")
print(f"Using {SHARED}")

## 2. Create the Model Services

In [0]:
from _lib.model_service_config import setup_model_service

model_service_fqns = {}
for service_name, service in MODEL_SERVICES.items():
    served_by = service["served_by"]
    model_service_fqns[service_name] = setup_model_service(
        catalog_name=SHARED_CATALOG,
        schema_name=SHARED_SCHEMA,
        service_name=service_name,
        served_by=served_by,
        comment=f"{service.get('comment', 'Routes to ' + served_by)} (Deploying and Monitoring Agents on Databricks)",
        recreate_on_drift=RECREATE_ON_DRIFT,
    )
    print(f"Model service: {model_service_fqns[service_name]} -> {served_by}")

## 3. Grant Access to All Users

Learners call the model services from notebooks and judges. `EXECUTE` on the schema covers every model service in it.

In [0]:
spark.sql(f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {SHARED} TO `account users`")
spark.sql(f"GRANT EXECUTE ON SCHEMA {SHARED} TO `account users`")
print(f"Granted EXECUTE on {SHARED} to account users")

## 4. Test the Model Services

In [0]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI(use_ai_gateway=True)
for fqn in model_service_fqns.values():
    response = client.chat.completions.create(
        model=fqn,
        messages=[{"role": "user", "content": "Reply with the word ready."}],
    )
    print(f"{fqn}: {response.choices[0].message.content}")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
for service_name, fqn in model_service_fqns.items():
    print(f"  Model service : {fqn} -> {MODEL_SERVICES[service_name]['served_by']}")